In [ ]:
!pip install -q langchain-classic langchain_openai langchain-core langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import os
with open("/content/api_key.txt") as archivo:
  apikey = archivo.read()
os.environ["OPENAI_API_KEY"] = apikey

In [3]:
from langchain_classic.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.schema.runnable import RunnableLambda, RunnableParallel, RunnableSequence

# Paso 1: Componentes individuales
limpiar = RunnableLambda(lambda x: x.strip())
mayus = RunnableLambda(lambda x: x.upper())
contar_palabras = RunnableLambda(lambda x: len(x.split()))

# Paso 2: Prompt de resumen
resumen_prompt = PromptTemplate.from_template("Resume brevemente este texto en una frase corta de maximo 10 palabras: {texto}")
modelo = ChatOpenAI(temperature=0)
parser = StrOutputParser()
resumen_chain = resumen_prompt | modelo | parser




In [4]:
# Paso 3: Flujo paralelo
analizador = RunnableParallel({
    "original_limpio": limpiar,
    "mayusculas": mayus,
    "palabras": contar_palabras,
    "resumen": resumen_chain,
})

# Paso 4: Combinar todo en un reporte
reporte = RunnableLambda(lambda x: f"📝 Resumen: {x['resumen']}\n🔢 Palabras: {x['palabras']}\n🧾 Original limpio: {x['original_limpio']}")

# Paso 5: Secuencia total
pipeline = RunnableSequence(analizador | reporte)



In [5]:
# Ejecutar
entrada = """   LangChain es una biblioteca muy poderosa y versátil que facilita el trabajo con modelos de
lenguaje, permitiendo construir aplicaciones más complejas como agentes, cadenas de procesamiento,
flujos conversacionales y sistemas basados en recuperación de información, todo de una manera estructurada y escalable.  """
print(pipeline.invoke(entrada))

📝 Resumen: LangChain facilita trabajo con modelos de lenguaje para aplicaciones complejas.
🔢 Palabras: 42
🧾 Original limpio: LangChain es una biblioteca muy poderosa y versátil que facilita el trabajo con modelos de
lenguaje, permitiendo construir aplicaciones más complejas como agentes, cadenas de procesamiento,
flujos conversacionales y sistemas basados en recuperación de información, todo de una manera estructurada y escalable.
